In [ ]:
import sys
import subprocess

print("Installing required libraries...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "gspread",
    "google-auth",
    "google-auth-oauthlib",
    "openai",
    "python-dotenv"
])

print("Required libraries installed successfully.")

Installing required libraries...
Required libraries installed successfully.


In [ ]:
import os
import json
import gspread
import google.auth

from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv
from google_auth_oauthlib.flow import InstalledAppFlow

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
import os
import importlib
from dotenv import load_dotenv

try:
    userdata = importlib.import_module("google.colab.userdata")
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except ModuleNotFoundError:
    load_dotenv()
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY:
    print("API key loaded")
else:
    print("API key not found")

API key loaded


In [31]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client configured successfully.")

OpenRouter client configured successfully.


In [ ]:
import gspread
import google.auth
from google_auth_oauthlib.flow import InstalledAppFlow

scopes = [
    "https://www.googleapis.com/auth/spreadsheets"
]

try:
    auth = __import__("google.colab.auth", fromlist=["authenticate_user"])

    auth.authenticate_user()
    creds, _ = google.auth.default()

    print("Google authentication completed in Colab.")

except ImportError:
    flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json",
        scopes
    )

    creds = flow.run_local_server(port=0)

    print("Google authentication completed in VS Code.")

sheets_client = gspread.authorize(creds)

print("Google Sheets client ready.")

In [ ]:
SPREADSHEET_ID = "1QTHrfg9dMz2KeG29jYTwD0Vo0iJrr13f25CK8jSZ4J8"

spreadsheet = sheets_client.open_by_key(SPREADSHEET_ID)

print("Google Sheet connected successfully.")
print("Available sheets:")

for sheet in spreadsheet.worksheets():
    print("-", sheet.title)

Google Sheet connected successfully.
Available sheets:
- Commission_Policy
- Sales
- Attendance_Policy
- Attendance
- Payroll


In [42]:
def number(value):
    try:
        return float(str(value).replace(",", "").replace("%", "").strip() or 0)
    except:
        return 0


def records(sheet_name):
    return spreadsheet.worksheet(sheet_name).get_all_records()


def get_attendance(employee_id):
    return [r for r in records("Attendance") if r["Employee ID"] == employee_id]


def get_sales(employee_id):
    return [r for r in records("Sales") if r["Employee ID"] == employee_id]


def get_attendance_policy():
    return records("Attendance_Policy")


def get_commission_policy():
    return records("Commission_Policy")


def get_payroll(employee_id=None):
    data = records("Payroll")
    return [r for r in data if not employee_id or r["Employee ID"] == employee_id]


def active_policy(policy, policy_type, day_type=None, as_of=None):
    result = []

    for row in policy:
        if row["Policy Type"] != policy_type:
            continue

        if str(row["Active"]).strip().lower() != "yes":
            continue

        if day_type and row["Day Type"] != day_type:
            continue

        effective_from = row.get("Effective From")

        if effective_from and as_of:
            try:
                effective_date = datetime.strptime(
                    str(effective_from).strip(),
                    "%Y-%m-%d"
                ).date()

                if effective_date > as_of:
                    continue

            except:
                pass

        result.append(row)

    return result


def attendance_pay(employee_id):
    attendance = get_attendance(employee_id)
    policy = get_attendance_policy()

    result = {
        "currency": policy[0]["Currency"] if policy else "",
        "overtime": 0,
        "late_entry": 0,
        "early_exit": 0,
        "violations": []
    }

    for row in attendance:
        day_type = row["Day Type"]

        try:
            attendance_date = datetime.strptime(
                str(row["Attendance Date"]).strip(),
                "%Y-%m-%d"
            ).date()
        except:
            attendance_date = None

        late = active_policy(
            policy,
            "Late Entry",
            day_type,
            attendance_date
        )

        early = active_policy(
            policy,
            "Early Exit",
            day_type,
            attendance_date
        )

        overtime = active_policy(
            policy,
            "Overtime",
            day_type,
            attendance_date
        )

        if late:
            p = late[-1]

            result["late_entry"] += (
                number(row["Late Entry Hours"])
                * number(p["Hourly Rate"])
                * number(p["Multiplier"])
            )

        if early:
            p = early[-1]

            result["early_exit"] += (
                number(row["Early Exit Hours"])
                * number(p["Hourly Rate"])
                * number(p["Multiplier"])
            )

        if overtime:
            p = overtime[-1]

            hours = number(row["Overtime Hours"])
            minimum = number(p["Daily Minimum Payable Hours"])
            maximum = number(p["Daily Maximum Payable Hours"])

            if hours >= minimum:
                payable_hours = min(hours, maximum)

                result["overtime"] += (
                    payable_hours
                    * number(p["Hourly Rate"])
                    * number(p["Multiplier"])
                )

                if hours > maximum:
                    result["violations"].append({
                        "date": row["Attendance Date"],
                        "day_type": day_type,
                        "actual_hours": hours,
                        "payable_hours": maximum,
                        "excess_hours": hours - maximum
                    })

    return result


def commission_pay(employee_id):
    sales = get_sales(employee_id)
    policy = get_commission_policy()

    total_sales = sum(
        number(row["Sales Amount"])
        for row in sales
    )

    for p in policy:
        minimum = number(p["From Sales Amount"])
        maximum = number(p["To Sales Amount"])

        if minimum <= total_sales <= maximum:
            rate = number(p["Commission Percent"])

            return {
                "sales": total_sales,
                "commission": total_sales * rate / 100,
                "rate": rate,
                "calculation_type": p["Calculation Type"]
            }

    return {
        "sales": total_sales,
        "commission": 0,
        "rate": 0,
        "calculation_type": ""
    }


def verify_payroll(employee_id):
    payroll = get_payroll(employee_id)

    if not payroll:
        return {"error": "Payroll record not found."}

    row = payroll[0]

    attendance = attendance_pay(employee_id)
    commission = commission_pay(employee_id)

    expected = {
        "commission": commission["commission"],
        "overtime": attendance["overtime"],
        "total_earnings": (
            commission["commission"]
            + attendance["overtime"]
        ),
        "late_deduction": attendance["late_entry"],
        "early_deduction": attendance["early_exit"],
        "total_deductions": (
            attendance["late_entry"]
            + attendance["early_exit"]
        )
    }

    actual = {
        "commission": number(row["Commission Earning"]),
        "overtime": number(row["Overtime Earning"]),
        "total_earnings": number(row["Total Earnings"]),
        "late_deduction": number(row["Late Entry Deduction"]),
        "early_deduction": number(
            row.get(
                "Early Exit Dedcution",
                row.get("Early Exit Deduction", 0)
            )
        ),
        "total_deductions": number(row["Total Deductions"])
    }

    differences = {
        key: actual[key] - expected[key]
        for key in expected
    }

    return {
        "employee_id": employee_id,
        "employee": row["Employee"],
        "salary_slip_id": row["Salary Slip ID"],
        "currency": attendance["currency"],
        "payroll": actual,
        "expected": expected,
        "differences": differences,
        "commission_sales": commission["sales"],
        "commission_rate": commission["rate"],
        "overtime_violations": attendance["violations"],
        "earnings_correct": all(
            actual[key] == expected[key]
            for key in [
                "commission",
                "overtime",
                "total_earnings"
            ]
        ),
        "deductions_correct": all(
            actual[key] == expected[key]
            for key in [
                "late_deduction",
                "early_deduction",
                "total_deductions"
            ]
        )
    }


def update_payroll(employee_id, approved=False):
    if not approved:
        return {
            "status": "Update blocked",
            "message": "Human approval is required."
        }

    result = verify_payroll(employee_id)

    if "error" in result:
        return result

    sheet = spreadsheet.worksheet("Payroll")
    data = sheet.get_all_records()

    for i, row in enumerate(data, start=2):
        if row["Employee ID"] == employee_id:
            expected = result["expected"]

            sheet.update(
                f"G{i}:L{i}",
                [[
                    expected["commission"],
                    expected["overtime"],
                    expected["total_earnings"],
                    expected["late_deduction"],
                    expected["early_deduction"],
                    expected["total_deductions"]
                ]]
            )

            return {
                "status": "Updated",
                "employee_id": employee_id
            }

    return {"error": "Payroll row not found."}

In [72]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_attendance",
            "description": "Get attendance records for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"}
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_sales",
            "description": "Get sales records for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"}
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_attendance_policy",
            "description": "Get attendance and overtime policies.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_commission_policy",
            "description": "Get commission policies.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verify_payroll",
            "description": "Verify payroll using attendance, sales, policies and payroll data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"}
                },
                "required": ["employee_id"]
            }
        }
    }
]

tool_registry = {
    "get_attendance": get_attendance,
    "get_sales": get_sales,
    "get_attendance_policy": get_attendance_policy,
    "get_commission_policy": get_commission_policy,
    "verify_payroll": verify_payroll
}

SYSTEM_PROMPT = """
You are a goal-driven AI Payroll Verification Agent.

Understand the user's goal and decide which available tools are needed.

Do not follow a fixed workflow.

Google Sheets is the source of truth.
Python verification results are authoritative.

Your role is to reason about the verification result and explain what it means.

Do not calculate, modify, or restate payroll numbers.
Do not create alternative numerical values.
Do not invent facts, policies, reasons, legal conclusions, or calculations.

When explaining a verification result:
- Explain what areas require attention.
- Explain the logical meaning of the discrepancies.
- Explain any overtime violations identified by Python.
- Base your reasoning only on information provided by the tools.
- If the available data does not establish a reason, say so.

Do not update payroll.
Payroll updates require explicit human approval.
"""

def format_python_result(result):
    print("\nPayroll Verification Result")
    print("---------------------------")
    print(f"Employee: {result['employee']}")
    print(f"Employee ID: {result['employee_id']}")
    print(f"Salary Slip: {result['salary_slip_id']}")
    print(f"Currency: {result['currency']}")

    print("\nCurrent Payroll:")
    for key, value in result["payroll"].items():
        print(f"{key}: {value}")

    print("\nExpected Payroll:")
    for key, value in result["expected"].items():
        print(f"{key}: {value}")

    print("\nDifferences:")
    for key, value in result["differences"].items():
        print(f"{key}: {value}")

    if result["overtime_violations"]:
        print("\nOvertime Violations:")
        for violation in result["overtime_violations"]:
            print(violation)

def run_agent(goal):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal}
    ]

    verification_result = None

    for _ in range(10):
        response = client.chat.completions.create(
            model="openrouter/free",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        if not message.tool_calls:
            if message.content:
                print("\nAgent Explanation")
                print("-----------------")
                print(message.content)

            if verification_result:
                format_python_result(verification_result)

            return

        messages.append(message)

        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = tool_registry[name](**args)

            if name == "verify_payroll":
                verification_result = result

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

    print("Agent stopped after reaching the maximum number of iterations.")

print("Goal-driven agent configured successfully.")

Goal-driven agent configured successfully.


In [79]:
def verify_and_approve(employee_id):
    result = verify_payroll(employee_id)

    if "error" in result:
        print(result["error"])
        return

    explanation_prompt = f"""
You are reviewing a payroll verification result.

Give a short logical explanation of what the result means.

Use only the information provided below.
Do not invent causes, policies, laws, or additional payroll concepts.
Do not recalculate any values.
Do not provide payroll numbers in your explanation.

Explain:
1. Whether earnings are correct.
2. Whether deductions are correct.
3. What the discrepancies indicate.
4. Whether any overtime violation was identified.
5. What should be reviewed before approval.

If the cause of a discrepancy is not available, explicitly say:
"The cause cannot be determined from the available data."

Verification result:
{json.dumps(result)}
"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "system",
                "content": "You are an AI payroll verification reasoning assistant."
            },
            {
                "role": "user",
                "content": explanation_prompt
            }
        ]
    )

    explanation = response.choices[0].message.content

    print("\nAgent Explanation")
    print("-----------------")
    print(explanation)

    format_python_result(result)

    approval = input(
        "\nApprove these corrections? (yes/no): "
    ).strip().lower()

    if approval == "yes":
        update_result = update_payroll(
            employee_id,
            approved=True
        )

        print("\nPayroll Update")
        print("--------------")
        print(update_result)

    else:
        print("\nUpdate cancelled. No changes were made.")


print("Verification, reasoning and human approval workflow ready.")

Verification, reasoning and human approval workflow ready.


In [80]:
verify_and_approve("EMP001")


Agent Explanation
-----------------
1. Earnings are correct: the result shows that commission, overtime, and total earnings have no differences from the expected values.  
2. Deductions are not correct: the result shows that the late deduction and early deduction items do not match the expected values.  
3. The discrepancies indicate that the individual deduction line items are inconsistent, even though the total deductions remain consistent with the expected total. The cause cannot be determined from the available data.  
4. An overtime violation was identified, indicating excess hours on a working day.  
5. Before approval, the late and early deduction discrepancies and the identified overtime violation should be reviewed and clarified.

Payroll Verification Result
---------------------------
Employee: Ahmed
Employee ID: EMP001
Salary Slip: SS-1
Currency: Rs

Current Payroll:
commission: 5000.0
overtime: 8200.0
total_earnings: 13200.0
late_deduction: 83.0
early_deduction: 43.0
total

C:\Users\muhammad.hunaid\AppData\Local\Temp\ipykernel_18496\1295814136.py:281: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(



Payroll Update
--------------
{'status': 'Updated', 'employee_id': 'EMP001'}
